This file contains my solution to a task from [Data Science Simulator](https://karpov.courses/simulator-ds), a hands-on training course in data analysis and machine learning by karpov.courses.

This file is shared with the permission of the course authors, and in a reduced form: the original problem statements and datasets are omitted. What's kept is my own code, reasoning, and commentary.

---

# Asymmetric loss functions

Two demand/value-forecasting settings where the cost of an error is asymmetric: being wrong in one direction hurts the business more than in the other. The goal is to pick a loss function whose penalty matches that asymmetry.

## Task 1. Demand forecasting

We forecast turnover for large household appliances to decide how much of each item to ship to a store; deliveries happen only once every few months. Under-shipping means lost sales and missed profit once stock runs out; over-shipping clogs the warehouse and blocks newer models. Which error is worse here, and what loss function reflects that?

### Solution

I use **RMSLE**, which penalizes under-prediction more heavily than over-prediction. When stock runs out the business earns nothing, and renting extra warehouse space is usually easier than securing an out-of-turn delivery (though this depends on the business). So over-shipping is the safer error — and RMSLE's asymmetry matches that preference.

In [ ]:
import numpy as np
from sklearn.metrics import mean_squared_log_error

def turnover_error(y_true: np.array, y_pred: np.array) -> float:
    """Calculate Root Mean Squared Logarithmic Error"""
    error = (mean_squared_log_error(y_true, y_pred))**(1/2)
    return error


## Task 2. LTV estimation

A B2B fintech startup with a small number of very large clients, each on a contract of at least a year. A model predicts a prospective client's LTV (lifetime value) to inform whether to sign them. Is it worse to under-estimate or over-estimate a client's value, and what loss captures that?

### Solution

Here it is better to **under-estimate** a client's value: with only a few clients, each one is a large share of revenue, so over-estimating risks locking the company into a long-term contract with a client who doesn't pay off.

I use the **LINEX** loss with `a = 0.8`. For `a > 0` it penalizes over-estimation exponentially while keeping the penalty for under-estimation roughly linear.

In [ ]:
import numpy as np

def ltv_error(y_true: np.array, y_pred: np.array, a: float = 0.8) -> float:
    """Calculate mean LINEX loss function

    a > 0: penalizes overestimation exponentially
    a < 0: penalizes underestimation exponentially

    """

    delta = y_pred - y_true
    error = np.mean(np.exp(a * delta) - a * delta - 1)

    return error